# UK Student Attainment Data Generator
**10,000 rows · UK National Averages 2024-25 · No modelling**

Columns: ref_id, SATS_score, GCSE_grade, GCE_AS_grade, Alevel_Maths_grade

In [ ]:
%pip install pandas numpy

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_rows = 10_000

AS_GRADES     = ["U", "E", "D", "C", "B", "A"]
ALEVEL_GRADES = ["U", "E", "D", "C", "B", "A", "A*"]
AS_TO_NUM     = {g: i for i, g in enumerate(AS_GRADES)}
ALEVEL_TO_NUM = {g: i for i, g in enumerate(ALEVEL_GRADES)}

ability = np.random.normal(0, 1, n_rows)
def to_unit(val, lo, hi): return (val - lo) / (hi - lo)

# SATS (UK national mean ~105, std ~8, range 80-120)
sats_raw = 105 + 8 * ability + np.random.normal(0, 3, n_rows)
sats = np.clip(np.round(sats_raw), 80, 120).astype(int)

# GCSE (national mean ~4.7, range 1-9)
gcse_raw = 4.7 + 1.5 * ability + np.random.normal(0, 0.8, n_rows)
gcse = np.clip(np.round(gcse_raw), 1, 9).astype(int)

# GCE AS-level (U-A; national peak at C/B)
as_raw = 2.8 + 1.3 * ability + np.random.normal(0, 0.7, n_rows)
as_idx = np.clip(np.round(as_raw), 0, 5).astype(int)
as_grades = np.array([AS_GRADES[i] for i in as_idx])

# A-level Maths — derived from SATS + GCSE + AS (3 inputs, no modelling)
composites = np.array([
    (to_unit(s, 80, 120) + to_unit(g, 1, 9) + to_unit(AS_TO_NUM[a], 0, 5)) / 3
    for s, g, a in zip(sats, gcse, as_grades)
])

# Percentile thresholds calibrated to UK A-level Maths 2024-25 national distribution
# A*=26%, A=17%, B=22%, C=18%, D=10%, E=5%, U=2%
p = np.percentile(composites, [2, 7, 17, 35, 57, 74])

def composite_to_maths_grade(c):
    if   c >= p[5]: return "A*"
    elif c >= p[4]: return "A"
    elif c >= p[3]: return "B"
    elif c >= p[2]: return "C"
    elif c >= p[1]: return "D"
    elif c >= p[0]: return "E"
    else:           return "U"

alevel_maths = np.array([composite_to_maths_grade(c) for c in composites], dtype=object)

ref_ids = np.random.permutation(np.arange(1_000_000, 1_000_000 + n_rows))

df = pd.DataFrame({
    "ref_id":             ref_ids,
    "SATS_score":         sats,
    "GCSE_grade":         gcse,
    "GCE_AS_grade":       as_grades,
    "Alevel_Maths_grade": alevel_maths,
})

df.to_csv("synthetic_uk_attainment_10000.csv", index=False)
print(df.head(10).to_string(index=False))
print(f"Saved {len(df):,} rows to synthetic_uk_attainment_10000.csv")
print(df["Alevel_Maths_grade"].value_counts(normalize=True).reindex(["A*","A","B","C","D","E","U"]).mul(100).round(1))
